# $\pi$: historical constructions as CMF trajectories

Each construction uses exact CMF transport and reduced rational readouts.
We report $\delta_N=-1-\log|\pi-p_N/q_N|/\log q_N$ at the stated source
index. Published measure bounds, convergence estimates, and measured
finite-depth exponents are distinct quantities.

The classical Gauss continued fraction and the 2005 Gregory-tail specialization
stop at source convergent 1000 (500 CMF steps). Mahler and Mignotte stop at
300, Bai at three periods; the other constructions run through 3000.
The shared helpers are in `cmfs.py`; standard hypergeometric fields come
directly from ramanujantools. Run cells in order to populate the comparison plot.


In [1]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 matplotlib==3.11.2

## Setup

Each section constructs a CMF, walks a trajectory with ramanujantools, reads
$p_N/q_N$ off the walk, and computes $\delta_N$. The walks and the
approximants are exact.

In [1]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx
from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

from cmfs import hata_pi, mignotte_pi, salikhov_pi, gauss_arctan_pi_seeds

sys.set_int_max_str_digits(0)
with ctx.workdps(92_000):  # q_N reaches about 45,000 digits at depth 3000
    PI = arb.pi()
DEPTHS = list(range(10, 3001, 10))
n = sp.Symbol("n")
R = sp.Rational
results = {}


def delta(approximant):
    """delta with |π - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |π - p/q|
        error = abs(PI - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

## 1. Classical Gauss arctangent continued fraction

[NIST DLMF 4.25.4](https://dlmf.nist.gov/4.25.E4) gives
$$\pi=\frac{4}{1+\frac{1^2}{3+\frac{2^2}{5+\frac{3^2}{7+\cdots}}}}.$$
This is also the $N=0,z=1$ specialization of equation (12) in
[Borwein–Choi–Pigulla (2005), *Continued Fractions of Tails of Hypergeometric Series*](https://www.sfu.ca/~schoia/paper/contfrac.pdf).
Let $R_M$ truncate after $M$ denominator entries, so $R_1=4$, $R_2=3$,
$R_3=19/6$, and $R_4=160/51$. The published consecutive-convergent error
estimate (proof of Theorem 4, p. 499), specialized to this positive continued
fraction, gives $|\pi-R_M|\le|R_M-R_{M-1}|$ for $M\ge2$.
No ordinary irrationality-measure upper bound is claimed here.

Use the full rank-two **three-axis** field `pFq(2, 1, -1)`, with axes
$(x_0,x_1;y_0)$, start $(\tfrac12,1;\tfrac32)$, and direction $(1,1;2)$.
For $W_k$ the walk of $k$ steps and $(w_0,w_1)^T=W_ke_0$,
$$R_{2k}=\frac{-2w_1}{2w_0-w_1}.$$
The seed identity is $F={}_2F_1(\tfrac12,1;\tfrac32;-1)=\pi/4$,
$\theta F=1/4-F/2$; the CMF transports its rational and $\pi$ coefficients.
We sample the source convergent index $M=10,20,\ldots,1000$;
**source index 1000 means 500 CMF steps**, not 1000 steps.


In [5]:
x0, x1, y0 = sp.symbols("x0 x1 y0")
cmf = pFq(2, 1, -1)
start = {x0: R(1, 2), x1: 1, y0: R(3, 2)}
trajectory = {x0: 1, x1: 1, y0: 2}
depths = list(range(10, 1001, 10))  # source convergent index M
P, Q = gauss_arctan_pi_seeds(0)
walks = cmf.walk(trajectory, [M // 2 for M in depths], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Gauss arctangent CF"] = (depths, deltas)
print(f"delta at source convergent 1000 (500 CMF steps): {deltas[-1]:.15f}")


delta at source convergent 1000 (500 CMF steps): -0.203168391923873


## 2. Mahler (1953): the same polynomial forms, of degree $10$

K. Mahler, *On the approximation of $\pi$*, Indag. Math. **15** (1953), 30–42,
[§§1–3](https://ems.press/books/dms/252/4986), applies the lemma of
K. Mahler, *On the approximation of logarithms of algebraic numbers*,
Phil. Trans. R. Soc. A **245** (1953), 371–398,
[Ch. 1, §§1–6](https://ems.press/books/dms/252/4985), with $m=10$ at
$x=i$, $\log x=i\pi/2$. With
$$Q(z)=\{z(z+1)\cdots(z+n)\}^{m+1},\qquad P=m!\,d_n^{\,m}(n!)^{m+1},$$
the forms are
$$R_h(x)=\frac{P}{2\pi i}\oint\frac{z^h x^{z+n}}{Q(z)}\,dz
=\sum_{k=0}^{m}A_{hk}(x)(\log x)^k,\qquad 0\le h\le m,$$
with $A_{hk}\in\mathbb Z[x]$ of degree at most $n$. These are the forms of
Section 2 with $m=10$ in place of Mignotte's $5$; Mahler takes $n\ge50$ and
needs $m+1\ge2|\log x|$.

**CMF:** `MeijerG(11, 0, 11, 11)` from `ramanujantools`, rank 11 over
$\mathbb Q(i)$, at the fibre $y=-i$, realified to rank 22, with the scalar
$$F_n(y)=G^{11,0}_{11,11}\!\left(y\;\middle|\;
\begin{matrix}n+1,\ldots,n+1\\0,\ldots,0\end{matrix}\right).$$
**Start:** $(1^{11};0^{11})$. **Direction:** $(1^{11};0^{11})$.
Depth $N$ evaluates Mahler's index $n=N$, and
$$R_h(i)=P\,i^n(-1)^h(\Theta^hF_n)(-i),\qquad \Theta=y\,d/dy,$$
so the state $(F,\Theta F,\ldots,\Theta^{10}F)$ is an invertible diagonal
transformation of Mahler's eleven forms.
Inverting the rank-11 $a$-shifts symbolically is slow, so `mahler_pi` in
`cmfs.py` returns the **dual** CMF directly: its axis matrices are the
transposed $a$-lowering matrices, which are polynomial, and its walk is the
inverse transpose of the walk. With $m=5$ the same function gives the dual
of the Mignotte CMF, and ramanujantools' symbolic conservation check passes.
At rank 22 in 11 variables that check does not finish in reasonable time,
so it is skipped. The supplementary script checks conservation exactly at
regular points instead.

The anchor state at $n=0$ is
$(\Theta^hF)(-i)=L^{10-h}/(10-h)!$ with $L=\log(-i)=-i\pi/2$, so, as in
Section 2, entries $[9,10]$ and $[10,10]$ of the dual walk have ratio
tending to $L$, and $\pi=\operatorname{Re}(2i\,x/y)$ is the extraction.
The walk and the extraction are exact. The supplementary script checks
exactly that the walk from the anchor state gives Mahler's $A_{hk}(i)$ for
$n=1,\ldots,8$.

**What it reaches:** $\delta_N\approx-0.54$. As for Mignotte, the scalar
route is not Mahler's: he never forms a rational approximation to $\pi$. He
substitutes $y=pi/(2q)$ for $\log x$ in a form $S_h=\sum_kA_{hk}(i)\,y^k$
that does not vanish. Then $(2q)^{10}S_h$ is a nonzero Gaussian integer,
while $R_h$ is tiny and $R_h-S_h=T_h\,(\log x-y)$ with $T_h$ bounded, which
bounds $|\pi-p/q|$ from below. That argument gives $\mu(\pi)\le42$ and is
not re-certified here.

**Cutoff: depth 300, not 3000.** As for Mignotte, $\delta_N$ is flat to two
decimals well before depth 300. Building the rank-22 CMF and its trajectory
matrix takes a few minutes; the walk itself takes seconds.

In [2]:
from cmfs import mahler_pi

a = sp.symbols("a0:11")
cmf = mahler_pi()  # already the dual CMF: its walks are inverse transposes of Mahler's walks
start = {x: 1 for x in a}
trajectory = {x: 1 for x in a}
depths = list(range(10, 301, 10))
dual = cmf.trajectory_matrix(trajectory, start)
duals = dual.walk({n: 1}, depths, {n: 0})
approximants = []
for D in duals:
    # Over Q(i), entries [9,10] = x and [10,10] = y of the inverse walk give pi = Re(2i x / y).
    x_re, x_im, y_re, y_im = D[9, 10], D[20, 10], D[10, 10], D[21, 10]
    approximants.append(2 * (x_re * y_im - x_im * y_re) / (y_re**2 + y_im**2))
deltas = [delta(r) for r in approximants]
results["Mahler (1953)"] = (depths, deltas)
print(f"delta at depth 300: {deltas[-1]:.9f}")

delta at depth 300: -0.537678259


## 3. Mignotte (1974): the Mahler--Mignotte polynomial forms

[Mignotte (1974), pp. 122--125](https://www.numdam.org/article/MSMF_1974__37__121_0.pdf)
uses contour forms in powers of $\log x$, and specializes to degree $5$
at $x=i$. The underlying unscaled scalar contour has the encoding
$$F_n(y)=G^{6,0}_{6,6}\!\left(y\;\middle|\;
\begin{matrix}n+1,n+1,n+1,n+1,n+1,n+1\\0,0,0,0,0,0\end{matrix}\right).$$
**CMF:** `MeijerG(6, 0, 6, 6)` from `ramanujantools`, rank 6 over
$\mathbb Q(i)$, at the fibre $y=-i$; `mignotte_pi` in `cmfs.py` realifies it to
rank 12, acting on rows $(\operatorname{Re}S,\operatorname{Im}S)$.
**Start:** $(1,1,1,1,1,1;0,0,0,0,0,0)$.
**Direction:** $(1,1,1,1,1,1;0,0,0,0,0,0)$.
With $\Theta=y\,d/dy$, Mignotte's normalization is
$$R_h(i)=P\,i^n(-1)^h(\Theta^hF_n)(-i),\qquad
P=5!\,d_n^5(n!)^6,\quad 0\le h\le5.$$
The derivative state is therefore an invertible diagonal transformation of
the historical forms. ramanujantools checks conservation when it builds
the CMF.

The forms are polynomial in $1,\pi,\ldots,\pi^5$ and give no scalar $p_N/q_N$
on their own, but the CMF's anchor state is exactly
$$(F,\Theta F,\ldots,\Theta^5F)
=\left(-\tfrac{L^5}{120},-\tfrac{L^4}{24},-\tfrac{L^3}{6},
-\tfrac{L^2}{2},-L,-1\right),\qquad L=\log(-i)=-\tfrac{i\pi}{2},$$
so the last two coordinates of the **dual** state have ratio $L$ itself. The
trajectory therefore does carry a scalar $p_N/q_N\to\pi$, and a $\delta_N$
defined exactly as in every other section. The walk and the extraction are exact.

**What it reaches:** $\delta_N\approx-0.41$, and the sign is the point. The
denominators grow about $4.7$ digits per step while the approximation gains
only $2.8$, so these approximants are *worse* than a generic rational of the
same height and carry no irrationality-measure information. This is not the
route by which Mignotte's bound is proved: that argument is a degree-$5$
transcendence-measure estimate on the forms, which is not re-certified here,
and neither is the notebook's Chudnovsky--Chudnovsky $14.65\ldots$ reference.

**Cutoff: depth 300, not 3000.** The rank-6 walk over $\mathbb Q(i)$ costs far
more per step than the rank-3 sections, and $\delta_N$ is flat to two decimals
well before 300.

In [3]:
a = sp.symbols("a0:6")
cmf = mignotte_pi()
start = {x: 1 for x in a}
trajectory = {x: 1 for x in a}
depths = list(range(10, 301, 10))
matrix = cmf.trajectory_matrix(trajectory, start)
dual = matrix.inverse().transpose()  # the dual walk is the inverse transpose of the walk
duals = dual.walk({n: 1}, depths, {n: 0})
approximants = []
for D in duals:
    # Over Q(i), entries [5,4] = x and [5,5] = y of the inverse walk give pi = Re(2i x / y).
    x_re, x_im, y_re, y_im = D[4, 5], D[10, 5], D[5, 5], D[11, 5]
    approximants.append(2 * (x_re * y_im - x_im * y_re) / (y_re**2 + y_im**2))
deltas = [delta(r) for r in approximants]
results["Mignotte (1974)"] = (depths, deltas)
print(f"delta at depth 300: {deltas[-1]:.9f}")

delta at depth 300: -0.412726235


## 4. Hata (1993)

M. Hata, *Rational approximations to $\pi$ and some other numbers*,
Acta Arith. **63** (1993), 335–349,
[Eqs. (1.3)–(1.4)](https://matwbn.icm.edu.pl/ksiazki/aa/aa63/aa6344.pdf).

**CMF:** rank 3 over $\mathbb Q(i)$ (rank 6 after realification), axes
$(u,v,w,m)$, for the moments $(J,zJ,z^2J)$, where this notation denotes
inserting $1,z,z^2$ into the integral
$$J(u,v,w,m)=\int_1^{1+i}
\frac{(z-1)^u(z-2)^v(z-1-i)^w}{z^m}\,dz.$$
**Start:** $(2,2,2,4)$ · **Direction:** $(2,2,2,3)$.
Depth $N$ evaluates Hata's historical index $n=N$.

Partial fractions give $J=C+R\log(1+i)$ with $C,R\in\mathbb Q(i)$.
The exact rational extraction is $-4\operatorname{Im}(C/R)$, since
$\operatorname{Im}\log(1+i)=\pi/4$. Both coefficient rows are transported
by the same CMF. `hata_pi` in `cmfs.py` builds the CMF and realifies it;
ramanujantools checks conservation.

The published bound is $\mu(\pi)<8.0161$. The calculation below recovers
the integral and rational approximants, not an independent proof of all
arithmetic and asymptotic estimates behind that bound.

In [4]:
u, v, w, m = sp.symbols("u v w m")
cmf = hata_pi()
start = {u: 2, v: 2, w: 2, m: 4}
trajectory = {u: 2, v: 2, w: 2, m: 3}
walks = cmf.walk(trajectory, [N - 1 for N in DEPTHS], start)  # the start is Hata's n = 1
# J = C + R log(1+i) at the start, as rows (Re J, Re zJ, Re z^2 J, Im J, Im zJ, Im z^2 J).
C = Matrix([[R(-50, 3), R(355, 12), R(-67, 3), 43, R(-118, 3), R(521, 30)]])
Rl = Matrix([[-38, 28, -8, -38, 50, -32]])
approximants = []
for W in walks:
    c, r = C * W, Rl * W
    approximants.append(-4 * (c[3] * r[0] - c[0] * r[3]) / (r[0]**2 + r[3]**2))  # -4 Im(C/R)
deltas = [delta(r) for r in approximants]
results["Hata (1993)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.144179100


## 5. Borwein–Choi–Pigulla (2005): Gauss approximation of a Gregory tail

[Equations (8), (10), (12) and Example 3](https://www.sfu.ca/~schoia/paper/contfrac.pdf)
apply Gauss's continued fraction after subtracting Gregory terms.
Here we choose the published family's fixed Taylor index $N=2$, $z=1$:
$$\pi=\frac83+\frac45\,{}_2F_1(\tfrac52,1;\tfrac72;-1),
\qquad R_M=\frac83+\frac45F_M(\tfrac52,1;\tfrac72;-1).$$
This is a specialization of their family, not a new historical record or
one of their tabulated large-$N$ experiments. It gives different rational
approximants: $R_2=47/15$ and $R_4=4288/1365$.
The published error bound specializes to
$|\pi-R_M|\le|R_M-R_{M-1}|$; no ordinary measure upper bound is asserted.

The same full rank-two, three-axis field has start
$(\tfrac52,1;\tfrac72)$ and direction $(1,1;2)$. Its readout is
$$R_{2k}=\frac{80w_0-230w_1}{30w_0-75w_1}.$$
Indeed $F=5\pi/4-10/3$ and $\theta F=5/4-(5/2)F$ supply the seed rows.
The plotted variable is the source CF index $M=2k$, with the Taylor index
fixed at $N=2$. Thus $\delta_{1000}$ again uses exactly 500 CMF steps.

The local verifier checks all even source convergents from 2 through 1000,
all three axis pairs at two regular points, the consecutive-convergent error
bound at each plotted sample, and the final Arb interval at two precisions.
These finite checks are recorded locally in the ignored `local_checks/` folder.


In [13]:
x0, x1, y0 = sp.symbols("x0 x1 y0")
cmf = pFq(2, 1, -1)
start = {x0: R(5, 2), x1: 1, y0: R(7, 2)}
trajectory = {x0: 1, x1: 1, y0: 2}
depths = list(range(10, 1001, 10))  # source convergent index M
P, Q = gauss_arctan_pi_seeds(2)
walks = cmf.walk(trajectory, [M // 2 for M in depths], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Borwein–Choi–Pigulla (2005), N=2"] = (depths, deltas)
print(f"delta at source convergent 1000 (500 CMF steps): {deltas[-1]:.15f}")


delta at source convergent 1000 (500 CMF steps): -0.197699778360353


## 6. Salikhov (2008/2010)

The original contour integral is displayed in
[Zeilberger–Zudilin, Part I](https://sites.math.rutgers.edu/~zeilberg/mamarim/mamarimPDF/pimeas.pdf).
After $t=x-5$, introduce
$$K(a,b,g)=-i\int_{-1-2i}^{-1+2i}
\frac{t^{2g}(t^4+6t^2+25)^a}{(t^2-25)^b}\,dt.$$

**CMF:** rank 3, axes $(a,b,g)$, row state
$(K(a,b,g),K(a,b,g+1),K(a,b,g+2))$.
**Start:** $(6,11,3)$ · **Direction:** $(6,10,3)$.
Depth $N$ evaluates the original Salikhov index $n=N$.
Section 5 uses the same CMF in the coordinates $I(A,B,G)=K(2A,2B,G)$.

Writing each moment as $C+R\pi$, the extraction is $-C/R$ from the first
moment. The matrices follow by polynomial reduction with
$$Q(s)=s^2+6s+25,\quad D(s)=sQ(s)(s-25),$$
$$\int\left[D r'+\left(D'+(g-\tfrac12)Q(s)(s-25)
+a sQ'(s)(s-25)-b sQ(s)\right)r\right]
s^{g-1/2}Q(s)^a(s-25)^{-b}\,ds=0.$$
The endpoint terms vanish. Reduction leaves the basis $(1,s,s^2)$.
`salikhov_pi` in `cmfs.py` builds the CMF and ramanujantools checks
conservation. The start moments below are exact partial-fraction values.
The published measure bound $7.606308$ is retained separately from the
computed exponent below.

In [5]:
a, b, g = sp.symbols("a b g")
cmf = salikhov_pi()
start = {a: 6, b: 11, g: 3}
trajectory = {a: 6, b: 10, g: 3}
walks = cmf.walk(trajectory, [N - 1 for N in DEPTHS], start)  # the start is Salikhov's n = 1
# K = C + R pi at the start, for (K(a,b,g), K(a,b,g+1), K(a,b,g+2)).
C = Matrix([[R(164534670966016, 39375), R(5446024280041216, 17325), R(21163235310750464, 1001)]])
Rp = Matrix([[R(-166263531392, 125), R(-500295344512, 5), -6729737285760]])
approximants = [-(C * W)[0] / (Rp * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Salikhov (2008/2010)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.156922525


## 7. Zeilberger–Zudilin (2020)

D. Zeilberger and W. Zudilin, *The irrationality measure of $\pi$ is at most
7.103205334137...*, Mosc. J. Comb. Number Theory **9** (2020), 407–419.
[Preprint](https://arxiv.org/abs/1912.06345).

**CMF:** $I(A,B,G)=K(2A,2B,G)$, the Salikhov CMF above ·
**Start:** $(1,2,1)$, that is $K(2,4,1)$ · **Direction:** $(2,3,2)$, that is
$(4,6,2)$ in $K$. Depth $N$ evaluates their integral $I_n$ with $n=2N+1$
[Eq. (1)](https://arxiv.org/abs/1912.06345), up to a constant factor.
The extraction is $-C/R$, as for Salikhov.

In [6]:
start = {a: 2, b: 4, g: 1}
trajectory = {a: 4, b: 6, g: 2}
walks = cmf.walk(trajectory, DEPTHS, start)
# K = C + R pi at the start.
C = Matrix([[R(11272, 15), R(658792, 15), R(74495016, 35)]])
Rp = Matrix([[R(-1196, 5), -13980, -677500]])
approximants = [-(C * W)[0] / (Rp * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Zeilberger–Zudilin (2020)"] = (DEPTHS, deltas)
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.163225345


## 8. Bai (2026 preprint): exact encoding, smaller computational cutoff

[Bai, Eqs. (1.1)–(1.8)](https://arxiv.org/html/2609.11276) uses
$(a,b,c)=(1857,3714,5570)$ in the preprint's notation.
In the moment CMF $K$ above this becomes

**CMF:** the same rank-3 moment CMF · **Start:** $(0,1,0)$ ·
**Direction:** $(3714,5570,1857)$.

Indeed the $n$th integrand is exactly
$$\frac{t^{3714n}(t^4+6t^2+25)^{3714n}}
{(t^2-25)^{5570n+1}}.$$
The rational extraction is again $-C/R$. The initial state is evaluated
exactly by partial fractions. The trajectory is too long for a symbolic
trajectory matrix, so each period is walked with `cmf.work`.

**Cutoff: 3 historical periods, not 3000.** Each period requires 11,141
elementary parameter shifts; the three points below use 33,423 shifts.
The depth convention is the historical integer $n$, and the plot labels
the actual cutoff. This verifies the encoding and gives finite numerical
evidence. It does not independently certify the preprint's bound
$\mu(\pi)<7.101862832357$.

In [7]:
start = {a: 0, b: 1, g: 0}
periods = [1, 2, 3]
walks = [cmf.work(start, {a: 3714 * k, b: 5570 * k + 1, g: 1857 * k}) for k in periods]
# K = C + R pi at the start.
C = Matrix([[0, 4, R(296, 3)]])
Rp = Matrix([[R(-1, 20), R(-5, 4), R(-125, 4)]])
approximants = [-(C * W)[0] / (Rp * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Bai (2026 preprint)"] = (periods, deltas)
print(f"delta at 3 periods: {deltas[-1]:.9f}")

delta at 3 periods: 0.164560474


## All constructions

The horizontal coordinate is each construction's stated source index; for the Gauss rows this is $M=2k$. Bai is plotted by period.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

$\delta_N$ is the approximation exponent the trajectory has actually reached
at its cutoff, printed by the cells above.  $\delta_{\mathrm{eq}}=1/(\mu_0-1)$
is the exponent equivalent to the published measure bound $\mu_0$, on a common
scale.  The two are different quantities: $\delta_N$ at a finite depth is
neither a bound, nor the limit the trajectory converges to.

| Construction | CMF | Initial point | Trajectory | cutoff | $\delta_N$ | $\mu_0$ | $\delta_{\mathrm{eq}}$ |
|:--|:--|:--|:--|--:|--:|--:|--:|
| Gauss arctangent CF | ${}_2F_1(-1)$ | $(1/2,1;3/2)$ | $(1,1;2)$ | $M=1000$ | $-0.203168391924$ | — | — |
| Borwein–Choi–Pigulla (2005), $N=2$ | ${}_2F_1(-1)$ | $(5/2,1;7/2)$ | $(1,1;2)$ | $M=1000$ | $-0.197699778360$ | — | — |
| Mahler (1953) | $G^{11,0}_{11,11}(-i)$ | $(1^{11};0^{11})$ | $(1^{11};0^{11})$ | 300 | $-0.54$ | $42$ | $0.02439$ |
| Mignotte (1974) | $G^{6,0}_{6,6}(-i)$ | $(1,1,1,1,1,1;0,0,0,0,0,0)$ | $(1,1,1,1,1,1;0,0,0,0,0,0)$ | 300 | $-0.41$ | $20$ | $0.05263$ |
| Hata (1993) | $J(u,v,w,m)$ | $(2,2,2,4)$ | $(2,2,2,3)$ | 3000 | $0.144179100$ | $8.0161$ | $0.14253$ |
| Salikhov (2008/2010) | $K(a,b,g)$ | $(6,11,3)$ | $(6,10,3)$ | 3000 | $0.156922525$ | $7.606308$ | $0.15137$ |
| Zeilberger--Zudilin (2020) | $I(A,B,G)$ | $(1,2,1)$ | $(2,3,2)$ | 3000 | $0.163225345$ | $7.103205334137\ldots$ | $0.163848$ |
| Bai (2026 preprint) | $K(a,b,g)$ | $(0,1,0)$ | $(3714,5570,1857)$ | 3 periods | $0.164560474$ | $7.101862832357\ldots$ | $0.163884$ |

For Mahler's and Mignotte's scalar readouts, $\delta_N<0$ is the reason: the dual
state of the Meijer-G field does converge to $\pi$, but far slower than its
denominators grow, so that trajectory approximates $\pi$ worse than a generic
rational of the same height. The published $\mu_0=42$ and $\mu_0=20$ rest on the
degree-$10$ and degree-$5$ forms instead, not on this scalar route. The four positive-exponent rows sit within a few
percent of the exponent equivalent to their published bounds.
The two Gauss rows recover approximation constructions with consecutive-convergent error bounds; neither supplies a measure bound. Their cutoffs use the source CF index $M$, twice the number of CMF steps.


## OpenAI (2026 preprint): reference only

Paper Table 1 records [the preprint](https://github.com/openai/math/blob/main/preprints/The-irrationality-exponent-of-pi-is-2-September-24-2026/paper.pdf)
as a published claim about $\mu(\pi)=2$ with an ineffective threshold.
This notebook has no CMF encoding, rational approximation sequence, or
finite-depth $\delta_N$ for that row. The paper table shows dashes in those
columns.
